# 🌱 Multi-Class Soil Image Classification (Task 1)
### **Soil Classification & Agronomic Intelligence 2025**
**Author:** Gogul Gupta (Team *Shree Yantra Dynamics*)  
**Affiliation:** Dr. A.P.J. Abdul Kalam Technical University  
**Leaderboard Result:** Public Score: **1.000** | Private Rank: **40**  

---
### 📌 Objective
Classify soil images into one of the four agricultural soil categories:
1. **Alluvial Soil**
2. **Black Soil**
3. **Clay Soil**
4. **Red Soil**

### 🧠 Modeling Strategy
- **Deep Learning Backbone:** ResNet-18 pretrained on ImageNet-1k
- **Evaluation Metric:** Minimum F1-Score across all 4 classes / Macro F1
- **Cross-Validation:** 5-Fold Stratified Cross-Validation

In [ ]:
# 📦 1. Imports and Environment Setup
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using compute device: {device}")

In [ ]:
# ⚙️ 2. Configuration & Paths
DATA_DIR = "data"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test")
TRAIN_CSV = os.path.join(DATA_DIR, "train_labels.csv")
TEST_CSV = os.path.join(DATA_DIR, "test_ids.csv")

CLASS_NAMES = ["Alluvial", "Black", "Clay", "Red"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
K_FOLDS = 5

In [ ]:
# 🔄 3. Transforms and Dataset Pipeline
train_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SoilDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = row["label"] if "label" in row else -1
        return image, label, row["image_id"]

In [ ]:
# 🏗️ 4. ResNet-18 Model Definition
def build_model(num_classes=4, pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

In [ ]:
# 🧪 5. Stratified 5-Fold Cross Validation Training Loop
if os.path.isfile(TRAIN_CSV):
    df_train = pd.read_csv(TRAIN_CSV)
    if df_train['soil_type'].dtype == object:
        class_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
        df_train['label'] = df_train['soil_type'].map(class_to_idx)
    
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    best_min_f1 = 0.0
    
    for fold, (t_idx, v_idx) in enumerate(skf.split(df_train, df_train['label'])):
        print(f"\n--- Fold {fold+1}/{K_FOLDS} ---")
        train_sub = df_train.iloc[t_idx].reset_index(drop=True)
        val_sub = df_train.iloc[v_idx].reset_index(drop=True)
        
        train_loader = DataLoader(SoilDataset(train_sub, TRAIN_IMG_DIR, train_transforms), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(SoilDataset(val_sub, TRAIN_IMG_DIR, val_transforms), batch_size=BATCH_SIZE, shuffle=False)
        
        model = build_model(num_classes=4, pretrained=True).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
        
        for epoch in range(EPOCHS):
            model.train()
            for imgs, labels, _ in train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()
        
        # Validate fold
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs = imgs.to(device)
                preds = model(imgs).argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(labels.numpy())
        
        fold_f1 = f1_score(val_targets, val_preds, average='macro')
        print(f"Fold {fold+1} Macro F1: {fold_f1:.4f}")
        print(classification_report(val_targets, val_preds, target_names=CLASS_NAMES))
else:
    print("ℹ️ Training CSV not found at default path. Load custom path to train.")

In [ ]:
# 🔮 6. Inference & Submission Generation
# Load trained checkpoint
model = build_model(num_classes=4, pretrained=False)
checkpoint_path = "models/best_model.pth" if os.path.exists("models/best_model.pth") else "best_model.pth"

if os.path.isfile(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model = model.to(device)
    model.eval()
    print(f"✅ Successfully loaded model from {checkpoint_path}")
else:
    print("⚠️ Checkpoint not found.")